## 3.1 Carga el CSV con read_csv

Importar dependencias necesarias

In [36]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine


Cargar el archivo csv , y mostrar la descripción de las columnas.

In [37]:
df_raw = pd.read_csv(
    "../data/ventas_techventas.csv",
    encoding="latin-1",
    parse_dates=["fecha"]
)
df_raw.info()
df = df_raw.copy()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     str           
 1   fecha            49 non-null     datetime64[us]
 2   cliente_id       49 non-null     str           
 3   cliente_nombre   49 non-null     str           
 4   region           49 non-null     str           
 5   producto         48 non-null     str           
 6   categoria        49 non-null     str           
 7   cantidad         49 non-null     float64       
 8   precio_unitario  49 non-null     float64       
 9   descuento        49 non-null     float64       
 10  vendedor         49 non-null     str           
dtypes: datetime64[us](1), float64(3), str(7)
memory usage: 5.3 KB


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
2,"1003,2024-01-10,C003,DataSolutions,Centro,Moni...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz


Observaciones
- La columna cantidad debería ser validada para evitar valores negativos o cero.
- precio_unitario debe ser positivo para garantizar coherencia en ingresos.
- order_id tiene un total de 60, los demás tienen 49, y en la fila 2 se logra identificar que se debe a un mal formateo o mala separación de los datos.
- La columna fecha fue correctamente convertida a tipo datetime gracias a $\text{parsedate}$

## 3.2 Detectar errores

Detectar cuantas valores nulos o mal formateados existen por columna

In [38]:
df.isna().sum()

order_id            0
fecha              11
cliente_id         11
cliente_nombre     11
region             11
producto           12
categoria          11
cantidad           11
precio_unitario    11
descuento          11
vendedor           11
dtype: int64

Se filtran los datos corruptos o mal formateados 

In [39]:
corruptos=df["order_id"].str.contains(",", na=False)
df_corruptos = df[corruptos]
df_corruptos

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
2,"1003,2024-01-10,C003,DataSolutions,Centro,Moni...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"1007,2024-01-20,C002,Innovatek,Sur,Monitor 27""...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,"1011,2024-02-02,C001,TechCorp SA,Norte,Monitor...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,"1017,2024-02-18,C006,NetPrime,Norte,Monitor 27...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,"1024,2024-03-10,C010,DeltaOps,Norte,Monitor 27...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30,"1031,2024-03-28,C003,DataSolutions,Centro,Moni...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,"1036,2024-04-12,C005,Sistemas Globales,Oeste,M...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,"1039,2024-04-20,C002,Innovatek,Sur,Monitor 27""...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,"1051,2024-05-22,C009,GammaDevs,Oeste,Monitor 2...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,"1055,2024-06-04,C004,CloudNet,Este,Monitor 27""...",NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Se realiza la corrección de los registros separando los datos que estaban agrupados en la columna order_id, usando la coma como delimitador para ubicar cada valor en su columna correspondiente.

In [40]:
df_split = df_corruptos["order_id"].str.split(",", expand=True)
df_split

,0,1,2,3,4,5,6,7,8,9,10
2,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5,350.00,0.05,Ana Lï¿½ï¿
6,1007,2024-01-20,C002,Innovatek,Sur,"Monitor 27""",Electronica,2,350.00,0.05,Ana Lï¿½ï¿
10,1011,2024-02-02,C001,TechCorp SA,Norte,"Monitor 27""",Electronica,4,350.00,0.05,Ana Lï¿½ï¿
16,1017,2024-02-18,C006,NetPrime,Norte,"Monitor 27""",Electronica,6,350.00,0.05,Luis Mora
23,1024,2024-03-10,C010,DeltaOps,Norte,"Monitor 27""",Electronica,3,350.00,0.05,Ana Lï¿½ï¿
30,1031,2024-03-28,C003,DataSolutions,Centro,"Monitor 27""",Electronica,3,350.00,0.05,Maria Torres
35,1036,2024-04-12,C005,Sistemas Globales,Oeste,"Monitor 27""",Electronica,4,350.00,0.05,Ana Lï¿½ï¿
38,1039,2024-04-20,C002,Innovatek,Sur,"Monitor 27""",Electronica,2,350.00,0.00,Maria Torres
50,1051,2024-05-22,C009,GammaDevs,Oeste,"Monitor 27""",Electronica,3,350.00,0.05,Maria Torres
54,1055,2024-06-04,C004,CloudNet,Este,"Monitor 27""",Electronica,5,350.00,0.05,Maria Torres


Volver a colocar los titulos de las columnas y convertir al tipo original las columnas: fecha,cantidad,precio_unitario,descuento

In [41]:
df_split.columns = df.columns
#Convertir tipos
df_split.info()
df_split.head(2)


<class 'pandas.DataFrame'>
Index: 11 entries, 2 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   order_id         11 non-null     str  
 1   fecha            11 non-null     str  
 2   cliente_id       11 non-null     str  
 3   cliente_nombre   11 non-null     str  
 4   region           11 non-null     str  
 5   producto         11 non-null     str  
 6   categoria        11 non-null     str  
 7   cantidad         11 non-null     str  
 8   precio_unitario  11 non-null     str  
 9   descuento        11 non-null     str  
 10  vendedor         11 non-null     str  
dtypes: str(11)
memory usage: 1.0 KB


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
2,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5,350.00,0.05,Ana Lï¿½ï¿
6,1007,2024-01-20,C002,Innovatek,Sur,"Monitor 27""",Electronica,2,350.00,0.05,Ana Lï¿½ï¿


Mantener las columnas que no presentaban ese problema o corrupción en el "order_id"

In [42]:
df_ok = df[~corruptos]
#Convertir tipos
df_ok = df_ok.astype(df_split.dtypes.to_dict())
df_ok.info()
df_ok.head(2)

<class 'pandas.DataFrame'>
Index: 49 entries, 0 to 58
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   order_id         49 non-null     str  
 1   fecha            49 non-null     str  
 2   cliente_id       49 non-null     str  
 3   cliente_nombre   49 non-null     str  
 4   region           49 non-null     str  
 5   producto         48 non-null     str  
 6   categoria        49 non-null     str  
 7   cantidad         49 non-null     str  
 8   precio_unitario  49 non-null     str  
 9   descuento        49 non-null     str  
 10  vendedor         49 non-null     str  
dtypes: str(11)
memory usage: 4.6 KB


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.1,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.0,Carlos Ruiz


Unir las columnas corregidas con las que ya estaban bien 

In [43]:
df_final = pd.concat([df_ok, df_split], ignore_index=True)

Ordenar el order_id 

In [44]:
df_final = df_final.sort_values("order_id").reset_index(drop=True)
df_final.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.1,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.0,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5,350.00,0.05,Ana Lï¿½ï¿
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.1,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz


Luego de formatear correctamente los datos, se verifica si existen nulos

In [45]:
df_nulos=df_final[(df_final.isna().any(axis=1))]
df_nulos


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,NaN,Electronica,2.0,350.0,0.05,Maria Torres


Antes de tomar la decisión de eliminar el registro, como estrategia se revisa si existen otros productos con el mismo precio_unitario (350), con el objetivo de identificar patrones que permitan inferir el valor faltante de manera consistente.

In [46]:
#Se toma el precio_unitario del producto nulo
precio_objetivo = df_nulos["precio_unitario"].iloc[0]
precio_objetivo
#Se buscan otros productos que tengan este mismo precio y que no coincida con el "nulo"
similares = df_final[
    (df_final["precio_unitario"] == precio_objetivo) &
    (df_final["producto"].notna())
]
similares

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor


Dado que no existen coincidencias en el dataset para ese precio_unitario, no es posible establecer una relación confiable que permita deducir el producto asociado. Por ello, se elimina este nulo.

In [47]:
df_final=df_final.dropna()

Se verifica que no exista ningun nulo y la eliminación correcta del registro, es decir, en total deberian existir 59 registros.

In [48]:
df_final.info()


<class 'pandas.DataFrame'>
Index: 59 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   order_id         59 non-null     str  
 1   fecha            59 non-null     str  
 2   cliente_id       59 non-null     str  
 3   cliente_nombre   59 non-null     str  
 4   region           59 non-null     str  
 5   producto         59 non-null     str  
 6   categoria        59 non-null     str  
 7   cantidad         59 non-null     str  
 8   precio_unitario  59 non-null     str  
 9   descuento        59 non-null     str  
 10  vendedor         59 non-null     str  
dtypes: str(11)
memory usage: 5.5 KB


Cambiar los tipos de datos a como estaban inicialmente

In [49]:
df_final = df_final.astype(df.dtypes.to_dict())
df_final.info()

<class 'pandas.DataFrame'>
Index: 59 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         59 non-null     str           
 1   fecha            59 non-null     datetime64[us]
 2   cliente_id       59 non-null     str           
 3   cliente_nombre   59 non-null     str           
 4   region           59 non-null     str           
 5   producto         59 non-null     str           
 6   categoria        59 non-null     str           
 7   cantidad         59 non-null     float64       
 8   precio_unitario  59 non-null     float64       
 9   descuento        59 non-null     float64       
 10  vendedor         59 non-null     str           
dtypes: datetime64[us](1), float64(3), str(7)
memory usage: 5.5 KB


Verificar y validar que no existan registros con cantidades negativas 

In [50]:
df_cant = df_final.loc[df_final['cantidad'] <= 0]
df_cant

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
47,1048,2024-05-15,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,-1.0,1200.0,0.0,Ana Lï¿½ï¿


Se cambia la cantidad negativa encontrada a una unidad minima 1

In [51]:
df_final.loc[df_final["cantidad"] <= 0, "cantidad"] = 1
df_final.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5.0,350.0,0.05,Ana Lï¿½ï¿
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz


In [52]:
df_final.loc[df_final['cantidad'] <= 0]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor


In [53]:
df_final.loc[df_final['precio_unitario'] <= 0]


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor


In [54]:
df_final.loc[df_final['descuento'] <0]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor


In [55]:
df_final["vendedor"] = df_final["vendedor"].replace({
    "Ana Lï¿½ï¿": "Ana López"
})
df_final["vendedor"].unique()

<StringArray>
['Ana López', 'Carlos Ruiz', 'Luis Mora', 'Maria Torres']
Length: 4, dtype: str

## 3.3 Crea columnas ingreso_bruto, ingreso_neto y mes.

Crear columna de "ingreso_bruto"

In [56]:
df_final["ingreso_bruto"] = df_final["cantidad"] * df_final["precio_unitario"]

Crear columna de "ingreso_neto"

In [57]:
df_final["ingreso_neto"] = df_final["ingreso_bruto"] * (1 - df_final["descuento"])

Crear columna "mes"

In [58]:
df_final["mes"] = df_final["fecha"].dt.to_period("M")

In [59]:
df_final.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor,ingreso_bruto,ingreso_neto,mes
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana López,2400.0,2160.0,2024-01
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz,375.0,375.0,2024-01
2,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5.0,350.0,0.05,Ana López,1750.0,1662.5,2024-01
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora,850.0,765.0,2024-01
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz,3600.0,3060.0,2024-01


Se realiza la agrupación por mes

In [60]:
resumen = df_final.groupby('mes').agg(ingreso_total=('ingreso_neto','sum'), n=
('order_id','count'))
resumen

,ingreso_total,n
mes,,
2024-01,13773.50,10
2024-02,13050.00,10
2024-03,14454.00,11
2024-04,16689.00,11
2024-05,13122.25,10
2024-06,11209.25,7


## 3.4 Concat para unir el resumen mensual con una tabla de metas.

In [61]:
df_metas = pd.DataFrame({
    "mes": ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"],
    "meta_mensual": [
        14500,
        14000,
        15000,
        17000,
        14000,
        12000
    ]
})
df_metas["mes"] = pd.to_datetime(df_metas["mes"]).dt.to_period("M")
df_metas

,mes,meta_mensual
0,2024-01,14500
1,2024-02,14000
2,2024-03,15000
3,2024-04,17000
4,2024-05,14000
5,2024-06,12000


Realizar el concat

In [62]:
df_resumen = resumen.merge(df_metas, on="mes", how="left")
df_resumen

,mes,ingreso_total,n,meta_mensual
0,2024-01,13773.50,10,14500
1,2024-02,13050.00,10,14000
2,2024-03,14454.00,11,15000
3,2024-04,16689.00,11,17000
4,2024-05,13122.25,10,14000
5,2024-06,11209.25,7,12000


Calcular % de cumplimiento

In [63]:
df_resumen["%_cumplimiento"] = (
    df_resumen["ingreso_total"] * 100 / df_resumen["meta_mensual"]
).round(2)

df_resumen

,mes,ingreso_total,n,meta_mensual,%_cumplimiento
0,2024-01,13773.50,10,14500,94.99
1,2024-02,13050.00,10,14000,93.21
2,2024-03,14454.00,11,15000,96.36
3,2024-04,16689.00,11,17000,98.17
4,2024-05,13122.25,10,14000,93.73
5,2024-06,11209.25,7,12000,93.41


In [64]:
df_resumen["mes"]=df_resumen["mes"].astype(str)
df_final["mes"]=df_final["mes"].astype(str)

Realizar el merge para obtener el dataset completo

In [65]:
df_final = df_resumen.merge(df_final, on="mes", how="left")

In [66]:
df_final.head()

,mes,ingreso_total,n,meta_mensual,%_cumplimiento,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor,ingreso_bruto,ingreso_neto
0,2024-01,13773.5,10,14500,94.99,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana López,2400.0,2160.0
1,2024-01,13773.5,10,14500,94.99,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz,375.0,375.0
2,2024-01,13773.5,10,14500,94.99,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5.0,350.0,0.05,Ana López,1750.0,1662.5
3,2024-01,13773.5,10,14500,94.99,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora,850.0,765.0
4,2024-01,13773.5,10,14500,94.99,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz,3600.0,3060.0


## 3.5 Exportar el DataFrame a sql

In [67]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../output/techventas.db")

## ventas

In [68]:
df_final.to_sql(
    "ventas",
    engine,
    if_exists="replace",
    index=False
)

59

Verifica leyendo de vuelta con read_sql('SELECT COUNT(*) FROM ventas',
engine)

In [69]:
final_query = """
    SELECT COUNT(*)
    FROM ventas;
"""

pd.read_sql(final_query,engine)

,COUNT(*)
0,59


In [70]:
final_query2 = """
    SELECT *
    FROM ventas;
"""

pd.read_sql(final_query2,engine)

,mes,ingreso_total,n,meta_mensual,%_cumplimiento,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor,ingreso_bruto,ingreso_neto
0,2024-01,13773.50,10,14500,94.99,1001,2024-01-05 00:00:00.000000,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2.0,1200.0,0.10,Ana López,2400.0,2160.00
1,2024-01,13773.50,10,14500,94.99,1002,2024-01-07 00:00:00.000000,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz,375.0,375.00
2,2024-01,13773.50,10,14500,94.99,1003,2024-01-10 00:00:00.000000,C003,DataSolutions,Centro,"Monitor 27""",Electronica,5.0,350.0,0.05,Ana López,1750.0,1662.50
3,2024-01,13773.50,10,14500,94.99,1004,2024-01-12 00:00:00.000000,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora,850.0,765.00
4,2024-01,13773.50,10,14500,94.99,1005,2024-01-15 00:00:00.000000,C004,CloudNet,Este,Laptop Pro 15,Electronica,3.0,1200.0,0.15,Carlos Ruiz,3600.0,3060.00
5,2024-01,13773.50,10,14500,94.99,1006,2024-01-18 00:00:00.000000,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres,1900.0,1900.00
6,2024-01,13773.50,10,14500,94.99,1007,2024-01-20 00:00:00.000000,C002,Innovatek,Sur,"Monitor 27""",Electronica,2.0,350.0,0.05,Ana López,700.0,665.00
7,2024-01,13773.50,10,14500,94.99,1008,2024-01-22 00:00:00.000000,C006,NetPrime,Norte,Router WiFi 6,Redes,8.0,120.0,0.00,Luis Mora,960.0,960.00
8,2024-01,13773.50,10,14500,94.99,1009,2024-01-25 00:00:00.000000,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12.0,95.0,0.10,Maria Torres,1140.0,1026.00
9,2024-01,13773.50,10,14500,94.99,1010,2024-01-28 00:00:00.000000,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,1.0,1200.0,0.00,Carlos Ruiz,1200.0,1200.00
